# 03 - Friction Measures Car (v5, generated 2026-10-13)

Fabric folder: Dev_CGP > Notebooks > 03 - EADM > Collaboration. Run order: 01 - Policy Renewals 30d, 02 - Friction Frame, then this notebook.
v2: renewal-event joins by equality on each code (v1's IN over two columns compared every episode with every event).
v4: A3.B1 quote cycles from the MFQ retrieve chain (dlk.mfq_quotequery RetrieveSessionToken), not stream events.
v5: branches entered on their own spine (happy-path steps per the flow sheets), not their friction or outcome
steps; R1B2 / R4cB1 reliable from renewals due July 2025 (validation pack).

The Car measures of the friction extract (spec v3.3) on the frame's episodes: one row per Car episode in
`stg.fe3m_car` with all 112 Car fields. Writes `stg.fe3m_*` only.

- Flags 1 / 0 on rows of their own episode type, blank elsewhere; blank (Unread) before the reliable-from date or
  when an eligible episode meets neither rule; branches blank while the episode is open on DataDate (Pending).
- A long call: queue time + ring time >= `WaitThresholdSeconds`, or ended without an agent answering.
- Not buildable, left blank: F5, R6, C5 (no source); A5.B1 (no A5.B1 events in the stream).
- C1 resolution uses the episode ending on M3 / M4a / M4c / M6, since M5 "MTA processed" has no events.
- Reliable-from dates are parameters: the spec's where given, otherwise the first full month of the source
  (recorded for DE4).


Parameters. Reliable-from dates apply to EpisodeStart: the spec's value where it gives one, otherwise the first full month of the source in the data (to record for DE4). RF_Calls: Genesys queues complete from Nov 2022 (F1, R2, R3, C1, C3, C4). RF_F2: A4.F1 from Nov 2022. RF_F3, RF_A3B1: spec, the Car quote engine (Jan 2024). RF_F4: A7.B1.4 from late Jul 2023. RF_R1: logins (R2a / R2.F1) from 2021. RF_R4: spec, renewal-window quotes (May 2026; also R2.B1). RF_R5: spec. RF_DocIntake: spec (A6.B1, R4.B2, M3.B1). RF_R1B2, RF_R4cB1: spec, Car extract renewals \*due\* from Jul 2025 (EpisodeStart = due - 60 \>= 2025-05-02; earlier months under DE4).

In [ ]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "ChillLH"
    }
}

In [1]:
Set WaitThresholdSeconds = 600 ;
Set PreSaleLookbackDays = 30 ;
Set RF_Calls = '2022-11-01' ;
Set RF_F2 = '2022-11-01' ;
Set RF_F3 = '2024-01-01' ;
Set RF_F4 = '2023-08-01' ;
Set RF_R1 = '2021-07-01' ;
Set RF_R4 = '2026-05-01' ;
Set RF_R5 = '2021-07-01' ;
Set RF_A3B1 = '2024-01-01' ;
Set RF_DocIntake = '2023-10-01' ;
Set RF_R1B2 = '2025-05-02' ;
Set RF_R4cB1 = '2025-05-02' ;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 14, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

... (4 outputs left out)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

Car episodes from the frame, with the window end used by "before DecisionDate" rules and an open flag.

In [2]:
CREATE OR REPLACE TABLE stg.fe3m_car_ep AS
SELECT e.*,
       least(e.EpisodeEnd, COALESCE(e.DecisionDate, e.EpisodeEnd)) AS win_end,
       CASE WHEN e.EpisodeEnd > e.DataDate THEN 1 ELSE 0 END          AS is_open
FROM   stg.fe3_episode e
WHERE  e.Product = 'Car'
;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Calls with their wait and connection, per conversation, measured on the queues of the line the call belongs to. Long = queue time + ring time \>= WaitThresholdSeconds, or no agent answered (abandoned / ended before connecting).

In [3]:
CREATE OR REPLACE TABLE stg.fe3m_calls AS
SELECT c.conversationId, c.scv_id, c.call_ts, c.call_date, c.line, c.queue_product,
       SUM(CASE WHEN g.purpose = 'acd' THEN COALESCE(g.totalAcdWaitDuration, 0) ELSE 0 END
           + COALESCE(g.totalAgentAlertDuration, 0)) / 1000.0                                   AS wait_s,
       MAX(CASE WHEN g.agentAnswered = 1 OR g.totalAgentTalkDuration > 0 THEN 1 ELSE 0 END)   AS connected,
       MAX(COALESCE(g.abandoned, 0))                                                          AS abandoned_flag
FROM   stg.fe3_calls c
JOIN   dlk.genesys_session_summary g ON g.conversationId = c.conversationId AND g.originatingDirection = 'inbound'
JOIN   stg.fe3_queue_map m ON m.queue_name = g.queueName AND m.spec_line = c.line
WHERE  c.scv_id IS NOT NULL
GROUP  BY c.conversationId, c.scv_id, c.call_ts, c.call_date, c.line, c.queue_product
;

CREATE OR REPLACE TABLE stg.fe3m_calls_x AS
SELECT *, CASE WHEN connected = 0 OR wait_s >= ${WaitThresholdSeconds} THEN 1 ELSE 0 END AS is_long
FROM   stg.fe3m_calls
;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 17, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

Car events used by the measures (distinct: identical copies count once).

In [4]:
CREATE OR REPLACE TABLE stg.fe3m_car_events AS
SELECT DISTINCT EventTypeId, EventDate, TRIM(SourcePolicyReference) AS pc, SourceQuoteReference AS qr,
       CASE WHEN SCV_Customer_Key > 0 THEN SCV_Customer_Key END AS scv
FROM   ods.episodeeventsnumberedfinal
WHERE  ProductType = 'motor' AND EventDate >= '2022-01-01' AND EventDate <= current_date()
AND   (EventTypeId IN ('A1', 'A3', 'A3.F1', 'A4a', 'A4b', 'A4.F1', 'A6', 'A7', 'A7.B1.4',
                       'R0.F1', 'R1a', 'R1b', 'R1.B2', 'R2a', 'R2.F1', 'R4', 'R4c', 'M3')
       OR EventTypeId LIKE 'A3.B1%' OR EventTypeId LIKE 'A6.B1%' OR EventTypeId LIKE 'R2.B1%'
       OR EventTypeId LIKE 'R4.B2%' OR EventTypeId LIKE 'M3.B1%')
;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 18, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Call measures: per episode, the calls of a line inside the measure's window.

**Notes**

- resolved: a call connected at or after the first long or abandoned call (including that call if answered)

In [5]:
CREATE OR REPLACE TABLE stg.fe3m_car_callagg AS
WITH w AS (
    SELECT e.EpisodeID, 'F1' AS m, c.*
    FROM   stg.fe3m_car_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'sales' AND c.queue_product = 'Motor'
           AND c.call_date BETWEEN date_sub(e.SaleDate, ${PreSaleLookbackDays}) AND e.SaleDate
    WHERE  e.EpisodeType = 'First year'
    UNION ALL
    SELECT e.EpisodeID, 'R2', c.*
    FROM   stg.fe3m_car_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'renewal' AND c.queue_product = 'Motor'
           AND c.call_date BETWEEN e.EpisodeStart AND e.win_end
    WHERE  e.EpisodeType = 'Renewal'
    UNION ALL
    SELECT e.EpisodeID, 'C1', c.*
    FROM   stg.fe3m_car_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'change' AND c.queue_product = 'Motor'
           AND c.call_date BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'Change'
    UNION ALL
    SELECT e.EpisodeID, 'C3', c.*
    FROM   stg.fe3m_car_ep e JOIN stg.fe3m_calls_x c
           ON  c.scv_id = e.scv_id AND c.line = 'document'
           AND c.call_date BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'Document request'
),
w2 AS (
    SELECT w.*, MIN(CASE WHEN is_long = 1 THEN call_ts END) OVER (PARTITION BY EpisodeID, m) AS first_long
    FROM   w
)
SELECT EpisodeID, m, COUNT(*) AS n, SUM(connected) AS n_conn, SUM(1 - connected) AS n_aband,
       CAST(ROUND(MAX(wait_s)) AS INT) AS max_wait, SUM(is_long) AS n_long, MAX(first_long) AS first_long,
       MAX(CASE WHEN connected = 1 AND call_ts >= first_long THEN 1 ELSE 0 END) AS resolved
FROM   w2 GROUP BY EpisodeID, m
;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

R3: first online session (login) per renewal episode, then whether a renewal-line call follows (date level).

In [6]:
CREATE OR REPLACE TABLE stg.fe3m_car_r3 AS
WITH l AS (
    SELECT e.EpisodeID, MIN(v.EventDate) AS first_login
    FROM   stg.fe3m_car_ep e JOIN stg.fe3m_car_events v
           ON v.scv = e.scv_id AND v.EventTypeId = 'R2a' AND v.EventDate BETWEEN e.EpisodeStart AND e.win_end
    WHERE  e.EpisodeType = 'Renewal'
    GROUP  BY e.EpisodeID
)
SELECT l.EpisodeID, l.first_login,
       MAX(CASE WHEN c.conversationId IS NOT NULL THEN 1 ELSE 0 END) AS called_after
FROM   l JOIN stg.fe3m_car_ep e ON e.EpisodeID = l.EpisodeID
LEFT JOIN stg.fe3m_calls_x c
       ON c.scv_id = e.scv_id AND c.line = 'renewal' AND c.queue_product = 'Motor'
       AND c.call_date >= l.first_login AND c.call_date <= e.win_end
GROUP  BY l.EpisodeID, l.first_login
;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 20, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Renewal episodes under each code their events can carry: the policy and, if different, its renewed code. (v2: an IN over two columns made Spark compare every episode with every event.)

In [7]:
CREATE OR REPLACE TABLE stg.fe3m_car_rcodes AS
SELECT EpisodeID, PolicyRef AS code, DueDate, EpisodeEnd, win_end FROM stg.fe3m_car_ep WHERE EpisodeType = 'Renewal'
UNION
SELECT EpisodeID, new_policy_code, DueDate, EpisodeEnd, win_end FROM stg.fe3m_car_ep
WHERE  EpisodeType = 'Renewal' AND new_policy_code IS NOT NULL AND new_policy_code <> PolicyRef
;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

A3.B1 quote retrieve / re-quote cycles from dlk.mfq_quotequery (Vijay, 2026-10-13): a retrieve creates a new QuoteQueryGuid and records the old one in RetrieveSessionToken (on one of the new guid's rows). Customer of the new or the old guid from the stream's quote events (98% found). Not from the stream's A3.B1.x events.

In [8]:
CREATE OR REPLACE TABLE stg.fe3m_mfq_retrieves AS
WITH r AS (
    SELECT CAST(QuoteQueryGuid AS STRING) AS g, MAX(CAST(RetrieveSessionToken AS STRING)) AS t,
           CAST(MIN(CreatedDateTime) AS DATE) AS d
    FROM   dlk.mfq_quotequery
    WHERE  RetrieveSessionToken IS NOT NULL AND TRIM(CAST(RetrieveSessionToken AS STRING)) <> ''
    AND    CreatedDateTime >= '2022-01-01'
    GROUP  BY CAST(QuoteQueryGuid AS STRING)
),
k AS (
    SELECT CAST(SourceQuoteReference AS STRING) AS q, MAX(SCV_Customer_Key) AS scv
    FROM   ods.episodeeventsnumberedfinal
    WHERE  ProductType = 'motor' AND EpisodeType = 'acquisition' AND SourceQuoteReference IS NOT NULL
    AND    SCV_Customer_Key > 0
    GROUP  BY CAST(SourceQuoteReference AS STRING)
)
SELECT r.g, r.t, r.d, COALESCE(k1.scv, k2.scv) AS scv
FROM   r LEFT JOIN k k1 ON k1.q = r.g LEFT JOIN k k2 ON k2.q = r.t
;

CREATE OR REPLACE TABLE stg.fe3m_car_a3b1 AS
SELECT e.EpisodeID, COUNT(*) AS n
FROM   stg.fe3m_car_ep e
JOIN   stg.fe3m_mfq_retrieves r
       ON r.scv = e.scv_id AND r.d BETWEEN date_sub(e.SaleDate, ${PreSaleLookbackDays}) AND e.SaleDate
WHERE  e.EpisodeType = 'First year'
GROUP  BY e.EpisodeID
;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 23, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

Event aggregates per episode. Quote-keyed first-year events by customer in the 30 days before the sale; policy-keyed events by the policy inside the episode (renewal events also under the renewed code); logins by customer.

**Notes**

- invitations, held letters and the payment steps carry derived dates (due - 60 / due date): a wider band
- `q AS (`: R4: per re-quote (entity_ref), whether it returned a price
- `SUM(CASE WHEN EventTypeId IN ('A6.B1', 'A6.B1.1', 'A6.B1.2', 'A6.B1 ...`: branch spine only
- `THEN 1 ELSE 0 END) AS r2b1_win_n,`: branch spine only
- `THEN 1 ELSE 0 END) AS r4b2_n,`: branch spine only
- `SUM(CASE WHEN EventTypeId IN ('M3.B1', 'M3.B1.1', 'M3.B1.2', 'M3.B1 ...`: branch spine only

In [9]:
CREATE OR REPLACE TABLE stg.fe3m_car_evagg AS
WITH x AS (
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_car_ep e JOIN stg.fe3m_car_events v
           ON  v.scv = e.scv_id AND v.EventTypeId IN ('A1', 'A3', 'A3.F1', 'A4a')
           AND v.EventDate BETWEEN date_sub(e.SaleDate, ${PreSaleLookbackDays}) AND e.SaleDate
    WHERE  e.EpisodeType = 'First year'
    UNION ALL
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_car_ep e JOIN stg.fe3m_car_events v
           ON  v.pc = e.PolicyRef
           AND (v.EventTypeId IN ('A4b', 'A4.F1', 'A6', 'A7', 'A7.B1.4') OR v.EventTypeId LIKE 'A6.B1%')
           AND v.EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'First year'
    UNION ALL
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_car_ep e JOIN stg.fe3m_car_events v
           ON  v.scv = e.scv_id AND v.EventTypeId IN ('R2a', 'R2.F1')
           AND v.EventDate BETWEEN e.EpisodeStart AND e.win_end
    WHERE  e.EpisodeType = 'Renewal'
    UNION ALL
    SELECT r.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_car_rcodes r JOIN stg.fe3m_car_events v
           ON  v.pc = r.code
           AND (v.EventTypeId IN ('R0.F1', 'R1a', 'R1b', 'R1.B2', 'R4', 'R4c') OR v.EventTypeId LIKE 'R2.B1%'
                OR v.EventTypeId LIKE 'R4.B2%')
           AND v.EventDate BETWEEN date_sub(r.DueDate, 90) AND date_add(r.EpisodeEnd, 16)
    UNION ALL
    SELECT e.EpisodeID, v.EventTypeId, v.EventDate, v.qr
    FROM   stg.fe3m_car_ep e JOIN stg.fe3m_car_events v
           ON  v.pc = e.PolicyRef AND (v.EventTypeId = 'M3' OR v.EventTypeId LIKE 'M3.B1%')
           AND v.EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd
    WHERE  e.EpisodeType = 'Change'
),
q AS (
    SELECT EpisodeID, qr,
           MAX(CASE WHEN EventTypeId = 'R2.B1.3' THEN 1 ELSE 0 END)    AS has_price,
           MAX(CASE WHEN EventTypeId = 'R2.B1.3.F1' THEN 1 ELSE 0 END) AS no_price,
           MIN(CASE WHEN EventTypeId = 'R2.B1.3.F1' THEN EventDate END) AS no_price_date
    FROM   x JOIN stg.fe3m_car_ep e USING (EpisodeID)
    WHERE  EventTypeId LIKE 'R2.B1%' AND qr IS NOT NULL AND EventDate <= e.win_end
    GROUP  BY EpisodeID, qr
),
qa AS (
    SELECT EpisodeID, SUM(has_price) AS rq_with_price,
           SUM(CASE WHEN no_price = 1 AND has_price = 0 THEN 1 ELSE 0 END) AS rq_without_price,
           MIN(CASE WHEN no_price = 1 AND has_price = 0 THEN no_price_date END) AS first_no_price
    FROM   q GROUP BY EpisodeID
)
SELECT x.EpisodeID,
       MAX(CASE WHEN EventTypeId = 'A1' THEN 1 ELSE 0 END)                              AS a1,
       MAX(CASE WHEN EventTypeId = 'A3' THEN 1 ELSE 0 END)                              AS a3,
       MAX(CASE WHEN EventTypeId = 'A3.F1' THEN 1 ELSE 0 END)                           AS a3f1,
       MAX(CASE WHEN EventTypeId = 'A4a' THEN 1 ELSE 0 END)                             AS a4a,
       MAX(CASE WHEN EventTypeId IN ('A4b', 'A4.F1') THEN 1 ELSE 0 END)                 AS a4b_or_wa,
       SUM(CASE WHEN EventTypeId LIKE 'A3.B1%' THEN 1 ELSE 0 END)                       AS a3b1_n,
       MAX(CASE WHEN EventTypeId = 'A6' THEN 1 ELSE 0 END)                              AS a6,
       SUM(CASE WHEN EventTypeId IN ('A6.B1', 'A6.B1.1', 'A6.B1.2', 'A6.B1.3', 'A6.B1.6') THEN 1 ELSE 0 END)   AS a6b1_n,
       MIN(CASE WHEN EventTypeId = 'A7' THEN EventDate END)                             AS a7_first,
       SUM(CASE WHEN EventTypeId = 'R2a' THEN 1 ELSE 0 END)                             AS logins_ok,
       SUM(CASE WHEN EventTypeId = 'R2.F1' THEN 1 ELSE 0 END)                           AS logins_failed,
       MIN(CASE WHEN EventTypeId = 'R2.F1' THEN EventDate END)                          AS first_failed,
       MAX(CASE WHEN EventTypeId = 'R2a' THEN EventDate END)                            AS last_ok,
       MAX(CASE WHEN EventTypeId = 'R0.F1' AND EventDate <= e.DueDate THEN 1 ELSE 0 END) AS held,
       MIN(CASE WHEN EventTypeId = 'R0.F1' AND EventDate <= e.DueDate THEN EventDate END) AS held_first,
       MAX(CASE WHEN EventTypeId IN ('R1a', 'R1b') AND EventDate <= e.DueDate THEN 1 ELSE 0 END) AS invited,
       MAX(CASE WHEN EventTypeId IN ('R1a', 'R1b') AND EventDate <= e.DueDate THEN EventDate END) AS invited_last,
       SUM(CASE WHEN EventTypeId = 'R1.B2' THEN 1 ELSE 0 END)                           AS r1b2_n,
       SUM(CASE WHEN EventTypeId LIKE 'R2.B1%' AND EventDate <= e.win_end THEN 1 ELSE 0 END) AS r2b1_n,
       SUM(CASE WHEN EventTypeId IN ('R2.B1', 'R2.B1.1', 'R2.B1.2', 'R2.B1.3', 'R2.B1.4') AND EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd
                THEN 1 ELSE 0 END)                                                       AS r2b1_win_n,
       MAX(CASE WHEN EventTypeId = 'R4' THEN 1 ELSE 0 END)                              AS r4,
       SUM(CASE WHEN EventTypeId = 'R4c' THEN 1 ELSE 0 END)                             AS r4c_n,
       SUM(CASE WHEN EventTypeId IN ('R4.B2', 'R4.B2.1', 'R4.B2.2', 'R4.B2.3', 'R4.B2.6') AND EventDate BETWEEN e.EpisodeStart AND e.EpisodeEnd
                THEN 1 ELSE 0 END)                                                       AS r4b2_n,
       MAX(CASE WHEN EventTypeId = 'M3' THEN 1 ELSE 0 END)                              AS m3,
       SUM(CASE WHEN EventTypeId IN ('M3.B1', 'M3.B1.1', 'M3.B1.2', 'M3.B1.3', 'M3.B1.6') THEN 1 ELSE 0 END)   AS m3b1_n,
       MAX(qa.rq_with_price) AS rq_with_price, MAX(qa.rq_without_price) AS rq_without_price,
       MAX(qa.first_no_price) AS first_no_price
FROM   x JOIN stg.fe3m_car_ep e USING (EpisodeID)
LEFT JOIN qa USING (EpisodeID)
GROUP  BY x.EpisodeID
;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 24, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Second-pass event facts: F4 chases after the first dispatch, and R4 / R1 / F4 resolutions.

**Notes**

- `WITH k AS (`: the codes to search: first-year episodes by policy; renewal episodes by policy and renewed code

In [10]:
CREATE OR REPLACE TABLE stg.fe3m_car_ev2 AS
SELECT a.EpisodeID,
       SUM(CASE WHEN v.EventTypeId = 'A7.B1.4' AND v.EventDate >= a.a7_first AND v.EventDate <= e.EpisodeEnd
                THEN 1 ELSE 0 END)                                                        AS chases,
       MIN(CASE WHEN v.EventTypeId = 'A7.B1.4' AND v.EventDate >= a.a7_first AND v.EventDate <= e.EpisodeEnd
                THEN v.EventDate END)                                                     AS first_chase
FROM   stg.fe3m_car_evagg a
JOIN   stg.fe3m_car_ep e USING (EpisodeID)
LEFT JOIN stg.fe3m_car_events v ON v.pc = e.PolicyRef AND v.EventTypeId IN ('A7.B1.4')
WHERE  e.EpisodeType = 'First year' AND a.a7_first IS NOT NULL
GROUP  BY a.EpisodeID
;

CREATE OR REPLACE TABLE stg.fe3m_car_ev3 AS
WITH k AS (
    SELECT a.EpisodeID, e.PolicyRef AS code, e.EpisodeEnd, e.win_end, b.first_chase, a.first_no_price
    FROM   stg.fe3m_car_evagg a JOIN stg.fe3m_car_ep e ON e.EpisodeID = a.EpisodeID
    LEFT JOIN stg.fe3m_car_ev2 b ON b.EpisodeID = a.EpisodeID
    WHERE  b.first_chase IS NOT NULL OR a.first_no_price IS NOT NULL
    UNION
    SELECT a.EpisodeID, r.code, r.EpisodeEnd, r.win_end, CAST(NULL AS DATE), a.first_no_price
    FROM   stg.fe3m_car_evagg a JOIN stg.fe3m_car_rcodes r ON r.EpisodeID = a.EpisodeID
    WHERE  a.first_no_price IS NOT NULL
)
SELECT k.EpisodeID,
       MAX(CASE WHEN v.EventTypeId = 'A7' AND v.EventDate > k.first_chase AND v.EventDate <= k.EpisodeEnd THEN 1 ELSE 0 END)
                                                                                          AS a7_repeat_after_chase,
       MAX(CASE WHEN v.EventTypeId = 'R2.B1.3' AND v.EventDate > k.first_no_price AND v.EventDate <= k.win_end
                THEN 1 ELSE 0 END)                                                       AS price_after_no_price
FROM   k
LEFT JOIN stg.fe3m_car_events v ON v.pc = k.code AND v.EventTypeId IN ('A7', 'R2.B1.3')
GROUP  BY k.EpisodeID
;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 26, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

The Car measures: one row per Car episode, all 112 Car fields in the spec's order.

In [11]:
CREATE OR REPLACE TABLE stg.fe3m_car AS
SELECT e.EpisodeID, e.EpisodeType,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(f1.n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_FirstYear_F1_SalesCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(f1.n, 0) > 0) AND (COALESCE(f1.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Car_FirstYear_F1_SalesCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND (NOT (COALESCE(f1.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Car_FirstYear_F1_SalesCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT ((COALESCE(f1.n, 0) > 0) AND (COALESCE(f1.n_long, 0) > 0)) THEN NULL ELSE CASE WHEN f1.resolved = 1 THEN 'Resolved' ELSE 'Unresolved' END END
           AS Car_FirstYear_F1_SalesCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.n END
           AS Car_FirstYear_F1_SalesCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.n_conn END
           AS Car_FirstYear_F1_SalesCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.n_aband END
           AS Car_FirstYear_F1_SalesCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(f1.n, 0) > 0) AND NOT (COALESCE(f1.n_long, 0) > 0) AND NOT (NOT (COALESCE(f1.n_long, 0) > 0)))) OR NOT (COALESCE(f1.n, 0) > 0) THEN NULL ELSE f1.max_wait END
           AS Car_FirstYear_F1_SalesCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F2} AND NOT (FALSE)) OR ((COALESCE(ev.a1, 0) = 1) AND NOT (COALESCE(ev.a4b_or_wa, 0) = 1) AND NOT (COALESCE(ev.a4a, 0) = 1))) THEN NULL WHEN COALESCE(ev.a1, 0) = 1 THEN 1 ELSE 0 END
           AS Car_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F2} AND NOT (FALSE)) OR ((COALESCE(ev.a1, 0) = 1) AND NOT (COALESCE(ev.a4b_or_wa, 0) = 1) AND NOT (COALESCE(ev.a4a, 0) = 1))) THEN NULL WHEN (COALESCE(ev.a1, 0) = 1) AND (COALESCE(ev.a4b_or_wa, 0) = 1) THEN 1 ELSE 0 END
           AS Car_FirstYear_F2_StartedOnlineFinishedByPhone_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F2} AND NOT (FALSE)) OR ((COALESCE(ev.a1, 0) = 1) AND NOT (COALESCE(ev.a4b_or_wa, 0) = 1) AND NOT (COALESCE(ev.a4a, 0) = 1))) THEN NULL WHEN (COALESCE(ev.a1, 0) = 1) AND NOT (COALESCE(ev.a4b_or_wa, 0) = 1) AND (COALESCE(ev.a4a, 0) = 1) THEN 1 ELSE 0 END
           AS Car_FirstYear_F2_StartedOnlineFinishedByPhone_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F3} AND NOT (FALSE)) OR ((COALESCE(ev.a1, 0) = 1) AND NOT (COALESCE(ev.a3f1, 0) = 1) AND NOT (NOT (COALESCE(ev.a3f1, 0) = 1)))) THEN NULL WHEN COALESCE(ev.a1, 0) = 1 THEN 1 ELSE 0 END
           AS Car_FirstYear_F3_NoPriceOnline_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F3} AND NOT (FALSE)) OR ((COALESCE(ev.a1, 0) = 1) AND NOT (COALESCE(ev.a3f1, 0) = 1) AND NOT (NOT (COALESCE(ev.a3f1, 0) = 1)))) THEN NULL WHEN (COALESCE(ev.a1, 0) = 1) AND (COALESCE(ev.a3f1, 0) = 1) THEN 1 ELSE 0 END
           AS Car_FirstYear_F3_NoPriceOnline_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F3} AND NOT (FALSE)) OR ((COALESCE(ev.a1, 0) = 1) AND NOT (COALESCE(ev.a3f1, 0) = 1) AND NOT (NOT (COALESCE(ev.a3f1, 0) = 1)))) THEN NULL WHEN (COALESCE(ev.a1, 0) = 1) AND NOT (COALESCE(ev.a3f1, 0) = 1) AND (NOT (COALESCE(ev.a3f1, 0) = 1)) THEN 1 ELSE 0 END
           AS Car_FirstYear_F3_NoPriceOnline_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4} AND NOT (FALSE)) OR ((ev.a7_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) THEN NULL WHEN ev.a7_first IS NOT NULL THEN 1 ELSE 0 END
           AS Car_FirstYear_F4_DocsChasedByCustomer_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4} AND NOT (FALSE)) OR ((ev.a7_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) THEN NULL WHEN (ev.a7_first IS NOT NULL) AND (COALESCE(c2.chases, 0) > 0) THEN 1 ELSE 0 END
           AS Car_FirstYear_F4_DocsChasedByCustomer_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4} AND NOT (FALSE)) OR ((ev.a7_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) THEN NULL WHEN (ev.a7_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND (NOT (COALESCE(c2.chases, 0) > 0)) THEN 1 ELSE 0 END
           AS Car_FirstYear_F4_DocsChasedByCustomer_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4} AND NOT (FALSE)) OR ((ev.a7_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) OR NOT ((ev.a7_first IS NOT NULL) AND (COALESCE(c2.chases, 0) > 0)) THEN NULL ELSE CASE WHEN c3.a7_repeat_after_chase = 1 THEN 'Resolved' ELSE 'Unknown' END END
           AS Car_FirstYear_F4_DocsChasedByCustomer_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4} AND NOT (FALSE)) OR ((ev.a7_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) OR NOT (ev.a7_first IS NOT NULL) THEN NULL ELSE COALESCE(c2.chases, 0) END
           AS Car_FirstYear_F4_DocsChasedByCustomer_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_F4} AND NOT (FALSE)) OR ((ev.a7_first IS NOT NULL) AND NOT (COALESCE(c2.chases, 0) > 0) AND NOT (NOT (COALESCE(c2.chases, 0) > 0)))) OR NOT (ev.a7_first IS NOT NULL) THEN NULL ELSE datediff(c2.first_chase, ev.a7_first) END
           AS Car_FirstYear_F4_DocsChasedByCustomer_DaysFromDispatchToFirstChase,
       CAST(NULL AS INT)
           AS Car_FirstYear_F5_LoanSetupChase_Eligible,
       CAST(NULL AS INT)
           AS Car_FirstYear_F5_LoanSetupChase_Affected,
       CAST(NULL AS INT)
           AS Car_FirstYear_F5_LoanSetupChase_Comparison,
       CAST(NULL AS STRING)
           AS Car_FirstYear_F5_LoanSetupChase_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1} AND NOT (FALSE)) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) THEN NULL WHEN COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R1_FailedLogin_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1} AND NOT (FALSE)) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) THEN NULL WHEN (COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND (COALESCE(ev.logins_failed, 0) > 0) THEN 1 ELSE 0 END
           AS Car_Renewal_R1_FailedLogin_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1} AND NOT (FALSE)) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) THEN NULL WHEN (COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND (NOT (COALESCE(ev.logins_failed, 0) > 0)) THEN 1 ELSE 0 END
           AS Car_Renewal_R1_FailedLogin_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1} AND NOT (FALSE)) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) OR NOT ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND (COALESCE(ev.logins_failed, 0) > 0)) THEN NULL ELSE CASE WHEN ev.last_ok >= ev.first_failed THEN 'Resolved' ELSE 'Unresolved' END END
           AS Car_Renewal_R1_FailedLogin_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1} AND NOT (FALSE)) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) OR NOT (COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) THEN NULL ELSE COALESCE(ev.logins_failed, 0) END
           AS Car_Renewal_R1_FailedLogin_FailedLogins,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1} AND NOT (FALSE)) OR ((COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) AND NOT (COALESCE(ev.logins_failed, 0) > 0) AND NOT (NOT (COALESCE(ev.logins_failed, 0) > 0)))) OR NOT (COALESCE(ev.logins_ok, 0) + COALESCE(ev.logins_failed, 0) > 0) THEN NULL ELSE COALESCE(ev.logins_ok, 0) END
           AS Car_Renewal_R1_FailedLogin_SuccessfulLogins,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(r2.n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R2_RenewalCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(r2.n, 0) > 0) AND (COALESCE(r2.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Car_Renewal_R2_RenewalCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND (NOT (COALESCE(r2.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Car_Renewal_R2_RenewalCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT ((COALESCE(r2.n, 0) > 0) AND (COALESCE(r2.n_long, 0) > 0)) THEN NULL ELSE CASE WHEN r2.resolved = 1 THEN 'Resolved' ELSE 'Unresolved' END END
           AS Car_Renewal_R2_RenewalCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.n END
           AS Car_Renewal_R2_RenewalCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.n_conn END
           AS Car_Renewal_R2_RenewalCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.n_aband END
           AS Car_Renewal_R2_RenewalCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(r2.n, 0) > 0) AND NOT (COALESCE(r2.n_long, 0) > 0) AND NOT (NOT (COALESCE(r2.n_long, 0) > 0)))) OR NOT (COALESCE(r2.n, 0) > 0) THEN NULL ELSE r2.max_wait END
           AS Car_Renewal_R2_RenewalCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND NOT (NOT (COALESCE(r3.called_after, 0) = 1)))) THEN NULL WHEN r3.first_login IS NOT NULL THEN 1 ELSE 0 END
           AS Car_Renewal_R3_OnlineThenCalled_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND NOT (NOT (COALESCE(r3.called_after, 0) = 1)))) THEN NULL WHEN (r3.first_login IS NOT NULL) AND (COALESCE(r3.called_after, 0) = 1) THEN 1 ELSE 0 END
           AS Car_Renewal_R3_OnlineThenCalled_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND NOT (NOT (COALESCE(r3.called_after, 0) = 1)))) THEN NULL WHEN (r3.first_login IS NOT NULL) AND NOT (COALESCE(r3.called_after, 0) = 1) AND (NOT (COALESCE(r3.called_after, 0) = 1)) THEN 1 ELSE 0 END
           AS Car_Renewal_R3_OnlineThenCalled_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4} AND NOT (FALSE)) OR ((COALESCE(ev.r2b1_n, 0) > 0) AND NOT (COALESCE(ev.rq_without_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_without_price, 0) > 0)))) THEN NULL WHEN COALESCE(ev.r2b1_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R4_RequoteNoPrice_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4} AND NOT (FALSE)) OR ((COALESCE(ev.r2b1_n, 0) > 0) AND NOT (COALESCE(ev.rq_without_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_without_price, 0) > 0)))) THEN NULL WHEN (COALESCE(ev.r2b1_n, 0) > 0) AND (COALESCE(ev.rq_without_price, 0) > 0) THEN 1 ELSE 0 END
           AS Car_Renewal_R4_RequoteNoPrice_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4} AND NOT (FALSE)) OR ((COALESCE(ev.r2b1_n, 0) > 0) AND NOT (COALESCE(ev.rq_without_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_without_price, 0) > 0)))) THEN NULL WHEN (COALESCE(ev.r2b1_n, 0) > 0) AND NOT (COALESCE(ev.rq_without_price, 0) > 0) AND (NOT (COALESCE(ev.rq_without_price, 0) > 0)) THEN 1 ELSE 0 END
           AS Car_Renewal_R4_RequoteNoPrice_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4} AND NOT (FALSE)) OR ((COALESCE(ev.r2b1_n, 0) > 0) AND NOT (COALESCE(ev.rq_without_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_without_price, 0) > 0)))) OR NOT ((COALESCE(ev.r2b1_n, 0) > 0) AND (COALESCE(ev.rq_without_price, 0) > 0)) THEN NULL ELSE CASE WHEN c3.price_after_no_price = 1 THEN 'Resolved' ELSE 'Unresolved' END END
           AS Car_Renewal_R4_RequoteNoPrice_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4} AND NOT (FALSE)) OR ((COALESCE(ev.r2b1_n, 0) > 0) AND NOT (COALESCE(ev.rq_without_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_without_price, 0) > 0)))) OR NOT (COALESCE(ev.r2b1_n, 0) > 0) THEN NULL ELSE COALESCE(ev.rq_with_price, 0) END
           AS Car_Renewal_R4_RequoteNoPrice_RequotesWithPrice,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4} AND NOT (FALSE)) OR ((COALESCE(ev.r2b1_n, 0) > 0) AND NOT (COALESCE(ev.rq_without_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_without_price, 0) > 0)))) OR NOT (COALESCE(ev.r2b1_n, 0) > 0) THEN NULL ELSE COALESCE(ev.rq_without_price, 0) END
           AS Car_Renewal_R4_RequoteNoPrice_RequotesWithoutPrice,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4} AND NOT (FALSE)) OR ((COALESCE(ev.r2b1_n, 0) > 0) AND NOT (COALESCE(ev.rq_without_price, 0) > 0) AND NOT (NOT (COALESCE(ev.rq_without_price, 0) > 0)))) OR NOT (COALESCE(ev.r2b1_n, 0) > 0) THEN NULL ELSE CAST(NULL AS INT) END
           AS Car_Renewal_R4_RequoteNoPrice_RequotesAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R5} AND NOT (FALSE)) OR ((TRUE) AND NOT (COALESCE(ev.held, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) THEN NULL WHEN TRUE THEN 1 ELSE 0 END
           AS Car_Renewal_R5_RenewalHeld_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R5} AND NOT (FALSE)) OR ((TRUE) AND NOT (COALESCE(ev.held, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) THEN NULL WHEN (TRUE) AND (COALESCE(ev.held, 0) = 1) THEN 1 ELSE 0 END
           AS Car_Renewal_R5_RenewalHeld_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R5} AND NOT (FALSE)) OR ((TRUE) AND NOT (COALESCE(ev.held, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) THEN NULL WHEN (TRUE) AND NOT (COALESCE(ev.held, 0) = 1) AND (COALESCE(ev.invited, 0) = 1) THEN 1 ELSE 0 END
           AS Car_Renewal_R5_RenewalHeld_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R5} AND NOT (FALSE)) OR ((TRUE) AND NOT (COALESCE(ev.held, 0) = 1) AND NOT (COALESCE(ev.invited, 0) = 1))) OR NOT ((TRUE) AND (COALESCE(ev.held, 0) = 1)) THEN NULL ELSE CASE WHEN ev.invited_last > ev.held_first THEN 'Resolved' ELSE 'Unresolved' END END
           AS Car_Renewal_R5_RenewalHeld_Resolution,
       CAST(NULL AS INT)
           AS Car_Renewal_R6_RenewalDateMissed_Eligible,
       CAST(NULL AS INT)
           AS Car_Renewal_R6_RenewalDateMissed_Affected,
       CAST(NULL AS INT)
           AS Car_Renewal_R6_RenewalDateMissed_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(c1.n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Change_C1_ChangeCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(c1.n, 0) > 0) AND (COALESCE(c1.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Car_Change_C1_ChangeCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND (NOT (COALESCE(c1.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Car_Change_C1_ChangeCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) OR NOT ((COALESCE(c1.n, 0) > 0) AND (COALESCE(c1.n_long, 0) > 0)) THEN NULL ELSE CASE WHEN e.match_evidence LIKE 'ended by M%' THEN 'Resolved' ELSE 'Unresolved' END END
           AS Car_Change_C1_ChangeCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) OR NOT (COALESCE(c1.n, 0) > 0) THEN NULL ELSE c1.n END
           AS Car_Change_C1_ChangeCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) OR NOT (COALESCE(c1.n, 0) > 0) THEN NULL ELSE c1.n_conn END
           AS Car_Change_C1_ChangeCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) OR NOT (COALESCE(c1.n, 0) > 0) THEN NULL ELSE c1.n_aband END
           AS Car_Change_C1_ChangeCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c1.n, 0) > 0) AND NOT (COALESCE(c1.n_long, 0) > 0) AND NOT (NOT (COALESCE(c1.n_long, 0) > 0)))) OR NOT (COALESCE(c1.n, 0) > 0) THEN NULL ELSE c1.max_wait END
           AS Car_Change_C1_ChangeCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) THEN NULL WHEN COALESCE(c3c.n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_DocRequest_C3_DocRequestCallWait_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n_long, 0) > 0) THEN 1 ELSE 0 END
           AS Car_DocRequest_C3_DocRequestCallWait_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND (NOT (COALESCE(c3c.n_long, 0) > 0)) THEN 1 ELSE 0 END
           AS Car_DocRequest_C3_DocRequestCallWait_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT ((COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n_long, 0) > 0)) THEN NULL ELSE CASE WHEN c3c.resolved = 1 THEN 'Resolved' ELSE 'Unresolved' END END
           AS Car_DocRequest_C3_DocRequestCallWait_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.n END
           AS Car_DocRequest_C3_DocRequestCallWait_CallsAttempted,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.n_conn END
           AS Car_DocRequest_C3_DocRequestCallWait_CallsConnected,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.n_aband END
           AS Car_DocRequest_C3_DocRequestCallWait_CallsAbandoned,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n_long, 0) > 0) AND NOT (NOT (COALESCE(c3c.n_long, 0) > 0)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE c3c.max_wait END
           AS Car_DocRequest_C3_DocRequestCallWait_LongestWaitSeconds,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) THEN NULL WHEN COALESCE(c3c.n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_DocRequest_C4_DocNotArrived_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n, 0) > 1) THEN 1 ELSE 0 END
           AS Car_DocRequest_C4_DocNotArrived_Affected,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) THEN NULL WHEN (COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND (NOT (COALESCE(c3c.n, 0) > 1)) THEN 1 ELSE 0 END
           AS Car_DocRequest_C4_DocNotArrived_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) OR NOT ((COALESCE(c3c.n, 0) > 0) AND (COALESCE(c3c.n, 0) > 1)) THEN NULL ELSE 'Unknown' END
           AS Car_DocRequest_C4_DocNotArrived_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'Document request' AND e.EpisodeStart >= ${RF_Calls} AND NOT (FALSE)) OR ((COALESCE(c3c.n, 0) > 0) AND NOT (COALESCE(c3c.n, 0) > 1) AND NOT (NOT (COALESCE(c3c.n, 0) > 1)))) OR NOT (COALESCE(c3c.n, 0) > 0) THEN NULL ELSE COALESCE(c3c.n, 0) - 1 END
           AS Car_DocRequest_C4_DocNotArrived_EventCount,
       CAST(NULL AS INT)
           AS Car_DocRequest_C5_DocFromInsurer_Eligible,
       CAST(NULL AS INT)
           AS Car_DocRequest_C5_DocFromInsurer_Affected,
       CAST(NULL AS INT)
           AS Car_DocRequest_C5_DocFromInsurer_Comparison,
       CAST(NULL AS STRING)
           AS Car_DocRequest_C5_DocFromInsurer_Resolution,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_A3B1}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.a3, 0) = 1) OR COALESCE(qc.n, 0) > 0) THEN 1 ELSE 0 END
           AS Car_FirstYear_A3B1_QuoteCycles_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_A3B1}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.a3, 0) = 1) OR COALESCE(qc.n, 0) > 0) AND COALESCE(qc.n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_FirstYear_A3B1_QuoteCycles_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_A3B1}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.a3, 0) = 1) OR COALESCE(qc.n, 0) > 0) AND NOT COALESCE(qc.n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_FirstYear_A3B1_QuoteCycles_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_A3B1}) OR e.is_open = 1) OR NOT ((COALESCE(ev.a3, 0) = 1) OR COALESCE(qc.n, 0) > 0) THEN NULL ELSE COALESCE(qc.n, 0) END
           AS Car_FirstYear_A3B1_QuoteCycles_EventCount,
       CAST(NULL AS INT)
           AS Car_FirstYear_A5B1_LoanSetup_Eligible,
       CAST(NULL AS INT)
           AS Car_FirstYear_A5B1_LoanSetup_Entered,
       CAST(NULL AS INT)
           AS Car_FirstYear_A5B1_LoanSetup_Comparison,
       CAST(NULL AS INT)
           AS Car_FirstYear_A5B1_LoanSetup_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.a6, 0) = 1) OR COALESCE(ev.a6b1_n, 0) > 0) THEN 1 ELSE 0 END
           AS Car_FirstYear_A6B1_DocIntakeAfterSale_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.a6, 0) = 1) OR COALESCE(ev.a6b1_n, 0) > 0) AND COALESCE(ev.a6b1_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_FirstYear_A6B1_DocIntakeAfterSale_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.a6, 0) = 1) OR COALESCE(ev.a6b1_n, 0) > 0) AND NOT COALESCE(ev.a6b1_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_FirstYear_A6B1_DocIntakeAfterSale_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'First year' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) OR NOT ((COALESCE(ev.a6, 0) = 1) OR COALESCE(ev.a6b1_n, 0) > 0) THEN NULL ELSE COALESCE(ev.a6b1_n, 0) END
           AS Car_FirstYear_A6B1_DocIntakeAfterSale_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1B2}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.invited, 0) = 1) OR COALESCE(ev.r1b2_n, 0) > 0) THEN 1 ELSE 0 END
           AS Car_Renewal_R1B2_AlternativeOffer_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1B2}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.invited, 0) = 1) OR COALESCE(ev.r1b2_n, 0) > 0) AND COALESCE(ev.r1b2_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R1B2_AlternativeOffer_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1B2}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.invited, 0) = 1) OR COALESCE(ev.r1b2_n, 0) > 0) AND NOT COALESCE(ev.r1b2_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R1B2_AlternativeOffer_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R1B2}) OR e.is_open = 1) OR NOT ((COALESCE(ev.invited, 0) = 1) OR COALESCE(ev.r1b2_n, 0) > 0) THEN NULL ELSE COALESCE(ev.r1b2_n, 0) END
           AS Car_Renewal_R1B2_AlternativeOffer_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.r2b1_win_n, 0) > 0) THEN 1 ELSE 0 END
           AS Car_Renewal_R2B1_OnlineRequote_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.r2b1_win_n, 0) > 0) AND COALESCE(ev.r2b1_win_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R2B1_OnlineRequote_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.r2b1_win_n, 0) > 0) AND NOT COALESCE(ev.r2b1_win_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R2B1_OnlineRequote_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4}) OR e.is_open = 1) OR NOT ((COALESCE(ev.logins_ok, 0) > 0) OR COALESCE(ev.r2b1_win_n, 0) > 0) THEN NULL ELSE COALESCE(ev.r2b1_win_n, 0) END
           AS Car_Renewal_R2B1_OnlineRequote_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.r4, 0) = 1) OR COALESCE(ev.r4b2_n, 0) > 0) THEN 1 ELSE 0 END
           AS Car_Renewal_R4B2_DocIntakeAfterRenewal_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.r4, 0) = 1) OR COALESCE(ev.r4b2_n, 0) > 0) AND COALESCE(ev.r4b2_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R4B2_DocIntakeAfterRenewal_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.r4, 0) = 1) OR COALESCE(ev.r4b2_n, 0) > 0) AND NOT COALESCE(ev.r4b2_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R4B2_DocIntakeAfterRenewal_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) OR NOT ((COALESCE(ev.r4, 0) = 1) OR COALESCE(ev.r4b2_n, 0) > 0) THEN NULL ELSE COALESCE(ev.r4b2_n, 0) END
           AS Car_Renewal_R4B2_DocIntakeAfterRenewal_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4cB1}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.r4, 0) = 1) OR COALESCE(ev.r4c_n, 0) > 0) THEN 1 ELSE 0 END
           AS Car_Renewal_R4cB1_NewMonthlyAgreement_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4cB1}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.r4, 0) = 1) OR COALESCE(ev.r4c_n, 0) > 0) AND COALESCE(ev.r4c_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R4cB1_NewMonthlyAgreement_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4cB1}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.r4, 0) = 1) OR COALESCE(ev.r4c_n, 0) > 0) AND NOT COALESCE(ev.r4c_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Renewal_R4cB1_NewMonthlyAgreement_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Renewal' AND e.EpisodeStart >= ${RF_R4cB1}) OR e.is_open = 1) OR NOT ((COALESCE(ev.r4, 0) = 1) OR COALESCE(ev.r4c_n, 0) > 0) THEN NULL ELSE COALESCE(ev.r4c_n, 0) END
           AS Car_Renewal_R4cB1_NewMonthlyAgreement_EventCount,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.m3, 0) = 1) OR COALESCE(ev.m3b1_n, 0) > 0) THEN 1 ELSE 0 END
           AS Car_Change_M3B1_DocIntakeForChange_Eligible,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.m3, 0) = 1) OR COALESCE(ev.m3b1_n, 0) > 0) AND COALESCE(ev.m3b1_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Change_M3B1_DocIntakeForChange_Entered,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) THEN NULL WHEN ((COALESCE(ev.m3, 0) = 1) OR COALESCE(ev.m3b1_n, 0) > 0) AND NOT COALESCE(ev.m3b1_n, 0) > 0 THEN 1 ELSE 0 END
           AS Car_Change_M3B1_DocIntakeForChange_Comparison,
       CASE WHEN (NOT (e.EpisodeType = 'Change' AND e.EpisodeStart >= ${RF_DocIntake}) OR e.is_open = 1) OR NOT ((COALESCE(ev.m3, 0) = 1) OR COALESCE(ev.m3b1_n, 0) > 0) THEN NULL ELSE COALESCE(ev.m3b1_n, 0) END
           AS Car_Change_M3B1_DocIntakeForChange_EventCount
FROM   stg.fe3m_car_ep e
LEFT JOIN stg.fe3m_car_evagg ev ON ev.EpisodeID = e.EpisodeID
LEFT JOIN stg.fe3m_car_ev2   c2 ON c2.EpisodeID = e.EpisodeID
LEFT JOIN stg.fe3m_car_ev3   c3 ON c3.EpisodeID = e.EpisodeID
LEFT JOIN stg.fe3m_car_r3    r3 ON r3.EpisodeID = e.EpisodeID
LEFT JOIN stg.fe3m_car_a3b1  qc ON qc.EpisodeID = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_car_callagg WHERE m = 'F1') f1  ON f1.EpisodeID  = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_car_callagg WHERE m = 'R2') r2  ON r2.EpisodeID  = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_car_callagg WHERE m = 'C1') c1  ON c1.EpisodeID  = e.EpisodeID
LEFT JOIN (SELECT * FROM stg.fe3m_car_callagg WHERE m = 'C3') c3c ON c3c.EpisodeID = e.EpisodeID
;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 27, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Checks

Grain (expect one row per Car episode) and the spec's invariants: an eligible episode is in exactly one group; comparison is 0 whenever affected is 1 (expect 0 violations).

In [12]:
SELECT (SELECT COUNT(*) FROM stg.fe3m_car) AS rows_, (SELECT COUNT(*) FROM stg.fe3m_car_ep) AS car_episodes,
       (SELECT COUNT(*) - COUNT(DISTINCT EpisodeID) FROM stg.fe3m_car) AS duplicate_ids;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 28, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

Per measure: rows filled, eligible, numerator, comparison, and invariant violations (expect bad_groups = 0).

In [13]:
SELECT 'Car_FirstYear_F1_SalesCallWait' AS measure, COUNT(Car_FirstYear_F1_SalesCallWait_Eligible) AS rows_filled, SUM(Car_FirstYear_F1_SalesCallWait_Eligible) AS eligible, SUM(Car_FirstYear_F1_SalesCallWait_Affected) AS numerator,
       SUM(Car_FirstYear_F1_SalesCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Car_FirstYear_F1_SalesCallWait_Eligible = 1 AND COALESCE(Car_FirstYear_F1_SalesCallWait_Affected, 0) + COALESCE(Car_FirstYear_F1_SalesCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_FirstYear_F2_StartedOnlineFinishedByPhone' AS measure, COUNT(Car_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible) AS rows_filled, SUM(Car_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible) AS eligible, SUM(Car_FirstYear_F2_StartedOnlineFinishedByPhone_Affected) AS numerator,
       SUM(Car_FirstYear_F2_StartedOnlineFinishedByPhone_Comparison) AS comparison,
       SUM(CASE WHEN Car_FirstYear_F2_StartedOnlineFinishedByPhone_Eligible = 1 AND COALESCE(Car_FirstYear_F2_StartedOnlineFinishedByPhone_Affected, 0) + COALESCE(Car_FirstYear_F2_StartedOnlineFinishedByPhone_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_FirstYear_F3_NoPriceOnline' AS measure, COUNT(Car_FirstYear_F3_NoPriceOnline_Eligible) AS rows_filled, SUM(Car_FirstYear_F3_NoPriceOnline_Eligible) AS eligible, SUM(Car_FirstYear_F3_NoPriceOnline_Affected) AS numerator,
       SUM(Car_FirstYear_F3_NoPriceOnline_Comparison) AS comparison,
       SUM(CASE WHEN Car_FirstYear_F3_NoPriceOnline_Eligible = 1 AND COALESCE(Car_FirstYear_F3_NoPriceOnline_Affected, 0) + COALESCE(Car_FirstYear_F3_NoPriceOnline_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_FirstYear_F4_DocsChasedByCustomer' AS measure, COUNT(Car_FirstYear_F4_DocsChasedByCustomer_Eligible) AS rows_filled, SUM(Car_FirstYear_F4_DocsChasedByCustomer_Eligible) AS eligible, SUM(Car_FirstYear_F4_DocsChasedByCustomer_Affected) AS numerator,
       SUM(Car_FirstYear_F4_DocsChasedByCustomer_Comparison) AS comparison,
       SUM(CASE WHEN Car_FirstYear_F4_DocsChasedByCustomer_Eligible = 1 AND COALESCE(Car_FirstYear_F4_DocsChasedByCustomer_Affected, 0) + COALESCE(Car_FirstYear_F4_DocsChasedByCustomer_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_FirstYear_F5_LoanSetupChase' AS measure, COUNT(Car_FirstYear_F5_LoanSetupChase_Eligible) AS rows_filled, SUM(Car_FirstYear_F5_LoanSetupChase_Eligible) AS eligible, SUM(Car_FirstYear_F5_LoanSetupChase_Affected) AS numerator,
       SUM(Car_FirstYear_F5_LoanSetupChase_Comparison) AS comparison,
       SUM(CASE WHEN Car_FirstYear_F5_LoanSetupChase_Eligible = 1 AND COALESCE(Car_FirstYear_F5_LoanSetupChase_Affected, 0) + COALESCE(Car_FirstYear_F5_LoanSetupChase_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Renewal_R1_FailedLogin' AS measure, COUNT(Car_Renewal_R1_FailedLogin_Eligible) AS rows_filled, SUM(Car_Renewal_R1_FailedLogin_Eligible) AS eligible, SUM(Car_Renewal_R1_FailedLogin_Affected) AS numerator,
       SUM(Car_Renewal_R1_FailedLogin_Comparison) AS comparison,
       SUM(CASE WHEN Car_Renewal_R1_FailedLogin_Eligible = 1 AND COALESCE(Car_Renewal_R1_FailedLogin_Affected, 0) + COALESCE(Car_Renewal_R1_FailedLogin_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Renewal_R2_RenewalCallWait' AS measure, COUNT(Car_Renewal_R2_RenewalCallWait_Eligible) AS rows_filled, SUM(Car_Renewal_R2_RenewalCallWait_Eligible) AS eligible, SUM(Car_Renewal_R2_RenewalCallWait_Affected) AS numerator,
       SUM(Car_Renewal_R2_RenewalCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Car_Renewal_R2_RenewalCallWait_Eligible = 1 AND COALESCE(Car_Renewal_R2_RenewalCallWait_Affected, 0) + COALESCE(Car_Renewal_R2_RenewalCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Renewal_R3_OnlineThenCalled' AS measure, COUNT(Car_Renewal_R3_OnlineThenCalled_Eligible) AS rows_filled, SUM(Car_Renewal_R3_OnlineThenCalled_Eligible) AS eligible, SUM(Car_Renewal_R3_OnlineThenCalled_Affected) AS numerator,
       SUM(Car_Renewal_R3_OnlineThenCalled_Comparison) AS comparison,
       SUM(CASE WHEN Car_Renewal_R3_OnlineThenCalled_Eligible = 1 AND COALESCE(Car_Renewal_R3_OnlineThenCalled_Affected, 0) + COALESCE(Car_Renewal_R3_OnlineThenCalled_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Renewal_R4_RequoteNoPrice' AS measure, COUNT(Car_Renewal_R4_RequoteNoPrice_Eligible) AS rows_filled, SUM(Car_Renewal_R4_RequoteNoPrice_Eligible) AS eligible, SUM(Car_Renewal_R4_RequoteNoPrice_Affected) AS numerator,
       SUM(Car_Renewal_R4_RequoteNoPrice_Comparison) AS comparison,
       SUM(CASE WHEN Car_Renewal_R4_RequoteNoPrice_Eligible = 1 AND COALESCE(Car_Renewal_R4_RequoteNoPrice_Affected, 0) + COALESCE(Car_Renewal_R4_RequoteNoPrice_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Renewal_R5_RenewalHeld' AS measure, COUNT(Car_Renewal_R5_RenewalHeld_Eligible) AS rows_filled, SUM(Car_Renewal_R5_RenewalHeld_Eligible) AS eligible, SUM(Car_Renewal_R5_RenewalHeld_Affected) AS numerator,
       SUM(Car_Renewal_R5_RenewalHeld_Comparison) AS comparison,
       SUM(CASE WHEN Car_Renewal_R5_RenewalHeld_Eligible = 1 AND COALESCE(Car_Renewal_R5_RenewalHeld_Affected, 0) + COALESCE(Car_Renewal_R5_RenewalHeld_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Renewal_R6_RenewalDateMissed' AS measure, COUNT(Car_Renewal_R6_RenewalDateMissed_Eligible) AS rows_filled, SUM(Car_Renewal_R6_RenewalDateMissed_Eligible) AS eligible, SUM(Car_Renewal_R6_RenewalDateMissed_Affected) AS numerator,
       SUM(Car_Renewal_R6_RenewalDateMissed_Comparison) AS comparison,
       SUM(CASE WHEN Car_Renewal_R6_RenewalDateMissed_Eligible = 1 AND COALESCE(Car_Renewal_R6_RenewalDateMissed_Affected, 0) + COALESCE(Car_Renewal_R6_RenewalDateMissed_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Change_C1_ChangeCallWait' AS measure, COUNT(Car_Change_C1_ChangeCallWait_Eligible) AS rows_filled, SUM(Car_Change_C1_ChangeCallWait_Eligible) AS eligible, SUM(Car_Change_C1_ChangeCallWait_Affected) AS numerator,
       SUM(Car_Change_C1_ChangeCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Car_Change_C1_ChangeCallWait_Eligible = 1 AND COALESCE(Car_Change_C1_ChangeCallWait_Affected, 0) + COALESCE(Car_Change_C1_ChangeCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_DocRequest_C3_DocRequestCallWait' AS measure, COUNT(Car_DocRequest_C3_DocRequestCallWait_Eligible) AS rows_filled, SUM(Car_DocRequest_C3_DocRequestCallWait_Eligible) AS eligible, SUM(Car_DocRequest_C3_DocRequestCallWait_Affected) AS numerator,
       SUM(Car_DocRequest_C3_DocRequestCallWait_Comparison) AS comparison,
       SUM(CASE WHEN Car_DocRequest_C3_DocRequestCallWait_Eligible = 1 AND COALESCE(Car_DocRequest_C3_DocRequestCallWait_Affected, 0) + COALESCE(Car_DocRequest_C3_DocRequestCallWait_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_DocRequest_C4_DocNotArrived' AS measure, COUNT(Car_DocRequest_C4_DocNotArrived_Eligible) AS rows_filled, SUM(Car_DocRequest_C4_DocNotArrived_Eligible) AS eligible, SUM(Car_DocRequest_C4_DocNotArrived_Affected) AS numerator,
       SUM(Car_DocRequest_C4_DocNotArrived_Comparison) AS comparison,
       SUM(CASE WHEN Car_DocRequest_C4_DocNotArrived_Eligible = 1 AND COALESCE(Car_DocRequest_C4_DocNotArrived_Affected, 0) + COALESCE(Car_DocRequest_C4_DocNotArrived_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_DocRequest_C5_DocFromInsurer' AS measure, COUNT(Car_DocRequest_C5_DocFromInsurer_Eligible) AS rows_filled, SUM(Car_DocRequest_C5_DocFromInsurer_Eligible) AS eligible, SUM(Car_DocRequest_C5_DocFromInsurer_Affected) AS numerator,
       SUM(Car_DocRequest_C5_DocFromInsurer_Comparison) AS comparison,
       SUM(CASE WHEN Car_DocRequest_C5_DocFromInsurer_Eligible = 1 AND COALESCE(Car_DocRequest_C5_DocFromInsurer_Affected, 0) + COALESCE(Car_DocRequest_C5_DocFromInsurer_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_FirstYear_A3B1_QuoteCycles' AS measure, COUNT(Car_FirstYear_A3B1_QuoteCycles_Eligible) AS rows_filled, SUM(Car_FirstYear_A3B1_QuoteCycles_Eligible) AS eligible, SUM(Car_FirstYear_A3B1_QuoteCycles_Entered) AS numerator,
       SUM(Car_FirstYear_A3B1_QuoteCycles_Comparison) AS comparison,
       SUM(CASE WHEN Car_FirstYear_A3B1_QuoteCycles_Eligible = 1 AND COALESCE(Car_FirstYear_A3B1_QuoteCycles_Entered, 0) + COALESCE(Car_FirstYear_A3B1_QuoteCycles_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_FirstYear_A5B1_LoanSetup' AS measure, COUNT(Car_FirstYear_A5B1_LoanSetup_Eligible) AS rows_filled, SUM(Car_FirstYear_A5B1_LoanSetup_Eligible) AS eligible, SUM(Car_FirstYear_A5B1_LoanSetup_Entered) AS numerator,
       SUM(Car_FirstYear_A5B1_LoanSetup_Comparison) AS comparison,
       SUM(CASE WHEN Car_FirstYear_A5B1_LoanSetup_Eligible = 1 AND COALESCE(Car_FirstYear_A5B1_LoanSetup_Entered, 0) + COALESCE(Car_FirstYear_A5B1_LoanSetup_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_FirstYear_A6B1_DocIntakeAfterSale' AS measure, COUNT(Car_FirstYear_A6B1_DocIntakeAfterSale_Eligible) AS rows_filled, SUM(Car_FirstYear_A6B1_DocIntakeAfterSale_Eligible) AS eligible, SUM(Car_FirstYear_A6B1_DocIntakeAfterSale_Entered) AS numerator,
       SUM(Car_FirstYear_A6B1_DocIntakeAfterSale_Comparison) AS comparison,
       SUM(CASE WHEN Car_FirstYear_A6B1_DocIntakeAfterSale_Eligible = 1 AND COALESCE(Car_FirstYear_A6B1_DocIntakeAfterSale_Entered, 0) + COALESCE(Car_FirstYear_A6B1_DocIntakeAfterSale_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Renewal_R1B2_AlternativeOffer' AS measure, COUNT(Car_Renewal_R1B2_AlternativeOffer_Eligible) AS rows_filled, SUM(Car_Renewal_R1B2_AlternativeOffer_Eligible) AS eligible, SUM(Car_Renewal_R1B2_AlternativeOffer_Entered) AS numerator,
       SUM(Car_Renewal_R1B2_AlternativeOffer_Comparison) AS comparison,
       SUM(CASE WHEN Car_Renewal_R1B2_AlternativeOffer_Eligible = 1 AND COALESCE(Car_Renewal_R1B2_AlternativeOffer_Entered, 0) + COALESCE(Car_Renewal_R1B2_AlternativeOffer_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Renewal_R2B1_OnlineRequote' AS measure, COUNT(Car_Renewal_R2B1_OnlineRequote_Eligible) AS rows_filled, SUM(Car_Renewal_R2B1_OnlineRequote_Eligible) AS eligible, SUM(Car_Renewal_R2B1_OnlineRequote_Entered) AS numerator,
       SUM(Car_Renewal_R2B1_OnlineRequote_Comparison) AS comparison,
       SUM(CASE WHEN Car_Renewal_R2B1_OnlineRequote_Eligible = 1 AND COALESCE(Car_Renewal_R2B1_OnlineRequote_Entered, 0) + COALESCE(Car_Renewal_R2B1_OnlineRequote_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Renewal_R4B2_DocIntakeAfterRenewal' AS measure, COUNT(Car_Renewal_R4B2_DocIntakeAfterRenewal_Eligible) AS rows_filled, SUM(Car_Renewal_R4B2_DocIntakeAfterRenewal_Eligible) AS eligible, SUM(Car_Renewal_R4B2_DocIntakeAfterRenewal_Entered) AS numerator,
       SUM(Car_Renewal_R4B2_DocIntakeAfterRenewal_Comparison) AS comparison,
       SUM(CASE WHEN Car_Renewal_R4B2_DocIntakeAfterRenewal_Eligible = 1 AND COALESCE(Car_Renewal_R4B2_DocIntakeAfterRenewal_Entered, 0) + COALESCE(Car_Renewal_R4B2_DocIntakeAfterRenewal_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Renewal_R4cB1_NewMonthlyAgreement' AS measure, COUNT(Car_Renewal_R4cB1_NewMonthlyAgreement_Eligible) AS rows_filled, SUM(Car_Renewal_R4cB1_NewMonthlyAgreement_Eligible) AS eligible, SUM(Car_Renewal_R4cB1_NewMonthlyAgreement_Entered) AS numerator,
       SUM(Car_Renewal_R4cB1_NewMonthlyAgreement_Comparison) AS comparison,
       SUM(CASE WHEN Car_Renewal_R4cB1_NewMonthlyAgreement_Eligible = 1 AND COALESCE(Car_Renewal_R4cB1_NewMonthlyAgreement_Entered, 0) + COALESCE(Car_Renewal_R4cB1_NewMonthlyAgreement_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car
UNION ALL
SELECT 'Car_Change_M3B1_DocIntakeForChange' AS measure, COUNT(Car_Change_M3B1_DocIntakeForChange_Eligible) AS rows_filled, SUM(Car_Change_M3B1_DocIntakeForChange_Eligible) AS eligible, SUM(Car_Change_M3B1_DocIntakeForChange_Entered) AS numerator,
       SUM(Car_Change_M3B1_DocIntakeForChange_Comparison) AS comparison,
       SUM(CASE WHEN Car_Change_M3B1_DocIntakeForChange_Eligible = 1 AND COALESCE(Car_Change_M3B1_DocIntakeForChange_Entered, 0) + COALESCE(Car_Change_M3B1_DocIntakeForChange_Comparison, 0) <> 1 THEN 1 ELSE 0 END) AS bad_groups
FROM stg.fe3m_car;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 29, Finished, Available, Finished, False)

<Spark SQL result set with 23 rows and 6 fields>

Resolution words by measure (only where affected = 1).

In [14]:
SELECT 'F1' m, Car_FirstYear_F1_SalesCallWait_Resolution r, COUNT(*) n FROM stg.fe3m_car WHERE Car_FirstYear_F1_SalesCallWait_Affected = 1 GROUP BY 2
UNION ALL SELECT 'F4', Car_FirstYear_F4_DocsChasedByCustomer_Resolution, COUNT(*) FROM stg.fe3m_car WHERE Car_FirstYear_F4_DocsChasedByCustomer_Affected = 1 GROUP BY 2
UNION ALL SELECT 'R1', Car_Renewal_R1_FailedLogin_Resolution, COUNT(*) FROM stg.fe3m_car WHERE Car_Renewal_R1_FailedLogin_Affected = 1 GROUP BY 2
UNION ALL SELECT 'R2', Car_Renewal_R2_RenewalCallWait_Resolution, COUNT(*) FROM stg.fe3m_car WHERE Car_Renewal_R2_RenewalCallWait_Affected = 1 GROUP BY 2
UNION ALL SELECT 'R5', Car_Renewal_R5_RenewalHeld_Resolution, COUNT(*) FROM stg.fe3m_car WHERE Car_Renewal_R5_RenewalHeld_Affected = 1 GROUP BY 2
UNION ALL SELECT 'C1', Car_Change_C1_ChangeCallWait_Resolution, COUNT(*) FROM stg.fe3m_car WHERE Car_Change_C1_ChangeCallWait_Affected = 1 GROUP BY 2
UNION ALL SELECT 'C3', Car_DocRequest_C3_DocRequestCallWait_Resolution, COUNT(*) FROM stg.fe3m_car WHERE Car_DocRequest_C3_DocRequestCallWait_Affected = 1 GROUP BY 2
ORDER BY 1, 2;

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 30, Finished, Available, Finished, False)

<Spark SQL result set with 13 rows and 3 fields>

In [15]:
%%pyspark
mssparkutils.session.stop()

StatementMeta(, 84d2e4ae-3496-409b-835c-e645a9ec0840, 32, Finished, Available, Finished, False)